# Physics-Augmented Graph Transformers for Patch-Antenna Forward and Inverse Design

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AviEpstein/GNN-for-Antenna-design/blob/public/notebooks/demo.ipynb)

**Avi Epstein, Snir Nehemia, Haim Suchowski, Lior Wolf** — Tel Aviv University.
IEEE MLSP 2026 · [code](https://github.com/AviEpstein/GNN-for-Antenna-design/tree/public) · [paper (PDF)](https://github.com/AviEpstein/GNN-for-Antenna-design/blob/public/docs/paper.pdf)

A patch antenna is described by a 16×16 metal mask on a grounded substrate. We turn that
geometry into a **surface mesh graph** and train a **GPS graph transformer** to predict its
34×34 far-field radiation pattern at 5.6 GHz — supervised not only on the pattern but also on
the **complex surface currents** flowing on the mesh (*PAIS*: Physics-Augmented Intermediate
Supervision), the physical intermediate that links geometry to radiation. For **inverse
design**, a conditional diffusion U-Net proposes candidate geometries for a target pattern and
the trained surrogate ranks them before CST validation. Ground truth everywhere comes from
full-wave CST simulation.

**This notebook has three tiers — stop whenever you like:**

| Tier | What you see | Runtime | Time |
|---|---|---|---|
| 1 | One antenna end-to-end + zero-shot interactive patch designer | free **CPU** | ~3 min |
| 2 | Reproduce Table-1-style metrics, PAIS ablation, direction-conditioned attention maps | free **CPU** | ~5 min |
| 3 | Inverse design: diffusion candidates ranked by the surrogate + denoising animation | **T4 GPU** | ~8 min |

Run cells top to bottom (`Runtime ▸ Run all` works for Tiers 1–2 on the default CPU runtime;
Tier 3 politely skips itself unless a GPU is attached).

In [ ]:
# --- constants: fill BUNDLE_BASE_URL after uploading the demo bundle (see scripts/build_demo_bundle.py)
import os
REPO_URL = os.environ.get('DEMO_REPO_URL', 'https://github.com/AviEpstein/GNN-for-Antenna-design.git')
REPO_BRANCH = os.environ.get('DEMO_REPO_BRANCH', 'public')
BUNDLE_BASE_URL = os.environ.get(
    'DEMO_BUNDLE_URL',
    'https://github.com/AviEpstein/GNN-for-Antenna-design/releases/download/demo-bundle-v1/')

from pathlib import Path
_root = next((c for c in [Path.cwd(), *Path.cwd().parents]
              if (c / 'configs' / 'base_forward.yaml').exists()), None)
if _root is None:                                            # not inside the repo -> clone it
    repo_name = REPO_URL.rstrip('/').removesuffix('.git').split('/')[-1]
    if not Path(repo_name).exists():
        !git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {repo_name}
    _root = Path(repo_name).resolve()
os.chdir(_root)
print('repo root:', Path.cwd())

In [ ]:
# --- dependencies: keep Colab's preinstalled torch, add PyG + companions from matching wheels
import importlib.util, os
os.environ['WANDB_MODE'] = 'disabled'   # the repo's Trainer would otherwise call wandb.init()

import ctypes.util, shutil as _shutil
if ctypes.util.find_library('GLU') is None and _shutil.which('apt-get'):
    # the gmsh binary links against libGLU, which Colab's headless image lacks
    print('installing system library libglu1-mesa (needed by gmsh)...')
    !apt-get -qq update > /dev/null && apt-get -qq install -y libglu1-mesa > /dev/null

_needed = ['torch_geometric', 'torch_scatter', 'torch_cluster',
           'pytorch_msssim', 'torchmetrics', 'trimesh', 'gmsh', 'cma']
_missing = [m for m in _needed if importlib.util.find_spec(m) is None]
if _missing:
    import torch, urllib.request
    _v = '.'.join(torch.__version__.split('+')[0].split('.')[:2])          # e.g. '2.8'
    _tag = f"cu{torch.version.cuda.replace('.', '')}" if (torch.version.cuda and torch.cuda.is_available()) else 'cpu'

    def _index_ok(url):
        try:
            return urllib.request.urlopen(url, timeout=10).status == 200
        except Exception:
            return False

    _candidates = [f'https://data.pyg.org/whl/torch-{_v}.0+{_tag}.html',
                   f'https://data.pyg.org/whl/torch-{_v}.0+cpu.html']
    _index = next((u for u in _candidates if _index_ok(u)), None)
    if _index is None:
        raise RuntimeError(
            f'No prebuilt torch-scatter/torch-cluster wheels for torch {torch.__version__}.\n'
            'Pin torch to a supported version and restart the runtime, e.g.:\n'
            '    %pip install torch==2.8.0 torchvision==0.23.0')
    print(f'installing {_missing}  (wheel index: {_index})')
    %pip install -q torch-geometric==2.6.1 torchmetrics==1.8.2 pytorch-msssim==1.0.0 trimesh==4.8.0 gmsh==4.14.0 cma==4.4.4 torch-scatter torch-cluster -f {_index}

import torch, torch_geometric
print(f'torch {torch.__version__} | torch_geometric {torch_geometric.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# --- download the demo bundle (checkpoints + sample data, ~100 MB). Cached: re-running is free.
import hashlib, json, tarfile, urllib.request
from pathlib import Path
from tqdm.auto import tqdm

if 'PLACEHOLDER' in BUNDLE_BASE_URL:
    raise RuntimeError('Set BUNDLE_BASE_URL in the constants cell above to where the demo '
                       'bundle is hosted (see scripts/build_demo_bundle.py for how it is built).')

_BDIR = Path('.demo_bundle'); _BDIR.mkdir(exist_ok=True)

def _download(name):
    dest = _BDIR / name
    if dest.exists():
        return dest
    url = BUNDLE_BASE_URL.rstrip('/') + '/' + name
    with tqdm(unit='B', unit_scale=True, desc=name) as bar:
        def _hook(nblocks, bs, total):
            if total > 0:
                bar.total = total
            bar.update(nblocks * bs - bar.n)
        urllib.request.urlretrieve(url, dest, reporthook=_hook)
    return dest

_manifest = json.loads(_download('manifest.json').read_text())

def fetch_and_extract(name):
    marker = _BDIR / (name + '.ok')
    if marker.exists():
        print(f'{name}: already downloaded and extracted')
        return
    dest = _download(name)
    if hashlib.sha256(dest.read_bytes()).hexdigest() != _manifest[name]['sha256']:
        dest.unlink()
        raise RuntimeError(f'{name}: checksum mismatch (corrupted download) — re-run this cell')
    with tarfile.open(dest) as tar:
        try:
            tar.extractall('.', filter='data')
        except TypeError:                      # Python without the tarfile filter argument
            tar.extractall('.')
    marker.touch()
    print(f"{name}: extracted ({_manifest[name]['size_bytes'] / 1e6:.0f} MB)")

for _name in ['demo_checkpoints.tar.gz', 'demo_data_tier1.tar.gz', 'demo_data_tier23.tar.gz']:
    fetch_and_extract(_name)

In [ ]:
%matplotlib inline
# --- shared helpers + load the headline GPS+PAIS (big) model (the paper's surrogate)
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch_geometric.transforms as T
from torch_geometric.data import Batch
from tqdm.auto import tqdm

from configs.parser import _load_yaml
from src.models.registry import build_graph_model
from src.graph.GNN_functions import prepare_graph
from src.losses.losses import compute_metrics

def load_config(row_yaml):
    cfg = _load_yaml('configs/base_forward.yaml')
    row = _load_yaml(f'configs/forward/{row_yaml}')
    cfg.update({k: v for k, v in row.items() if k != 'base_config'})
    return cfg

DEVICE = torch.device('cpu')                      # Tiers 1-2 run comfortably on CPU
CFG = load_config('zeroshot_classic_square.yaml') # GPS+PAIS graph/model settings
FF_STATS = pd.read_csv('data/corpora/dataset_ff_stats.csv').iloc[0].to_dict()
PE_TRANSFORM = T.Compose([T.ToUndirected(), T.AddLaplacianEigenvectorPE(k=10, attr_name='pe')])
IDX_FREQ = CFG['idx_freq']                        # 3 -> 5.6 GHz

def load_forward_model(ckpt, cfg=None):
    cfg = cfg or CFG
    model = build_graph_model(cfg, DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model.eval()

def graph_from_sample(ex, cfg=None):
    """Processed sample dict -> model-ready graph (mirrors PixelDataset.get)."""
    g, _ = prepare_graph(ex['graph'].clone(), cfg or CFG)
    g.x = torch.cat([g.pos, g.node_normals, g.node_type], dim=-1)
    return g

def predict(model, g, cfg=None):
    """Returns (radiation pattern [34,34], surface currents [N,6] or None)."""
    with torch.no_grad():
        out = model(Batch.from_data_list([g]),
                    radiation_image_shape=(cfg or CFG)['radiation_image_shape'])
    if isinstance(out, dict):
        return out['radiation_image'].squeeze(-1).squeeze(0), out.get('surface_current')
    return out.squeeze(-1).squeeze(0), None

def metrics_of(pred, label):
    """Paper Table-1 metric path. NOTE: the 'SNR' key is the paper's 'PSNR' column."""
    m = compute_metrics(pred[None, None], label[None, None], FF_STATS)
    return {k: float(v) for k, v in m.items()}

MODEL = load_forward_model('trained_models/genial-bush-2194GNN_ff_foward_34_34_2400.pt')
print(f'GPS+PAIS (big) loaded on {DEVICE}: '
      f'{sum(p.numel() for p in MODEL.parameters()):,} parameters. Setup OK.')

## Tier 1 — see the model work

**What the input is.** Each antenna is a 16×16 pixel mask: metal pixels of a 28 mm patch
region sitting on a 50 mm ground plane, separated by a 4 mm substrate (ε_r = 3.55). A coax
feed is placed at the brightest mask pixel. Some antennas add a second, *parasitic* metal
layer 3 mm above the patch. The geometry is meshed and turned into a graph: nodes carry
position, surface normal and a component type (patch / feed / ground / substrate / parasitic),
edges follow the mesh plus a radius graph.

**What the model predicts.** A 34×34 map of linear directivity over (θ, φ) at 5.6 GHz — and,
through the PAIS head, the complex surface current at every mesh node. Ground truth for both
comes from CST full-wave simulation. Below: the full anatomy of one parasitic-patch example.

In [ ]:
%matplotlib inline
# --- one antenna: masks, mesh graph, CST pattern, CST surface currents
from mpl_toolkits.mplot3d.art3d import Line3DCollection

EX = torch.load('data/corpora/classic_patch_with_reflector/processed/processed_data_0.pt',
                weights_only=False)
g_raw, ant = EX['graph'], EX['example_paramters']['ant_parameters']
print(f"mesh graph: {g_raw.num_nodes} nodes, {g_raw.num_edges} edges | "
      f"patch {ant['patch_x']} mm on {ant['ground_x']} mm ground, "
      f"h = {ant['h']} mm, eps_r = {ant['eps_r']}, parasitic layer at +{ant['reflector_distance']} mm")

fig = plt.figure(figsize=(18, 4))
ax = fig.add_subplot(151)
ax.imshow(ant['matrix'], cmap='gray'); ax.axis('off')
ax.set_title('patch mask (16x16)\nbrightest pixel = feed')
ax = fig.add_subplot(152)
ax.imshow(ant['reflector_matrix'], cmap='gray'); ax.axis('off')
ax.set_title('parasitic mask (16x16)')

ax = fig.add_subplot(153, projection='3d')
p, e = g_raw.pos.numpy(), g_raw.edge_index.numpy()
ax.add_collection3d(Line3DCollection(np.stack([p[e[0]], p[e[1]]], 1),
                                     colors='lightgray', linewidths=0.2))
ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=g_raw.node_type.argmax(1), cmap='tab10', s=4)
ax.set_title('mesh graph (3D stack)'); ax.set_box_aspect((1, 1, 0.5)); ax.set_axis_off()

ax = fig.add_subplot(154)
im = ax.imshow(EX['farfeilds'][IDX_FREQ], cmap='jet', vmin=0, vmax=10); ax.axis('off')
ax.set_title('CST pattern @ 5.6 GHz\n(linear directivity)')
plt.colorbar(im, ax=ax, fraction=0.046)

ax = fig.add_subplot(155, projection='3d')
pc = EX['pos_surface_current'][IDX_FREQ].numpy()
mag = EX['surface_currents'][IDX_FREQ].norm(dim=1).numpy()
ax.scatter(pc[:, 0], pc[:, 1], pc[:, 2], c=mag, cmap='inferno', s=2)
ax.set_title('CST surface currents |J|'); ax.set_box_aspect((1, 1, 0.5)); ax.set_axis_off()
plt.tight_layout(); plt.show()

In [ ]:
%matplotlib inline
# --- forward prediction: the whole paper in one figure
# We predict the plain square-patch sibling of the antenna above (the parasitic variant
# reappears in the interactive cell below and in the zero-shot benchmark).
EX_SQ = torch.load('data/corpora/classic_patch_no_reflector/processed/processed_data_0.pt',
                   weights_only=False)
g = graph_from_sample(EX_SQ)
t0 = time.time(); pred, pred_J = predict(MODEL, g); dt = time.time() - t0
label = EX_SQ['farfeilds'][IDX_FREQ]
m = metrics_of(pred, label)
print(f"inference: {dt*1000:.0f} ms on CPU | MSE {m['MSE']:.4f} | MAE {m['MAE']:.4f} | "
      f"MS-SSIM {m['mssim']:.4f} | PSNR {m['SNR']:.2f} dB   "
      f"(a CST run of the same antenna takes minutes-to-hours)")

fig = plt.figure(figsize=(18, 4))
for i, (img, title) in enumerate([(label, 'CST ground truth'), (pred, 'GPS+PAIS prediction'),
                                  ((pred - label).abs(), 'absolute error')]):
    ax = fig.add_subplot(1, 5, i + 1)
    im = ax.imshow(img, cmap='jet', vmin=0, vmax=(10 if i < 2 else 3)); ax.axis('off')
    ax.set_title(title); plt.colorbar(im, ax=ax, fraction=0.046)

ax = fig.add_subplot(154, projection='3d')
pc = EX_SQ['pos_surface_current'][IDX_FREQ].numpy()
mag = EX_SQ['surface_currents'][IDX_FREQ].norm(dim=1).numpy()
ax.scatter(pc[:, 0], pc[:, 1], pc[:, 2], c=mag, cmap='inferno', s=2)
ax.set_title('CST currents |J|'); ax.set_box_aspect((1, 1, 0.5)); ax.set_axis_off()

ax = fig.add_subplot(155, projection='3d')
pm = g.pos.numpy(); magp = pred_J.norm(dim=1).numpy()
ax.scatter(pm[:, 0], pm[:, 1], pm[:, 2], c=magp, cmap='inferno', s=4)
ax.set_title('PAIS-predicted currents |J|\n(at mesh nodes)')
ax.set_box_aspect((1, 1, 0.5)); ax.set_axis_off()
plt.tight_layout(); plt.show()

### Zero-shot on textbook patches

The model was trained **only** on FMNIST/CIFAR-silhouette pixel masks — it has never seen a
clean rectangular patch. Evaluating on canonically constructed square patches is therefore a
zero-shot generalization test. Published zero-shot row (square patches, n = 64):
**MAE 0.153 · MSE 0.056 · MS-SSIM 0.971 · PSNR 24.8 dB** (the printed 0.25 MAE in the paper is
a transcription error — see REPRODUCIBILITY.md, caveat 14). The parasitic variant is much
harder: **MAE 0.400 · MSE 0.447 · MS-SSIM 0.867 · PSNR 14.5 dB**.

Design a patch below — position, feed edge, optional parasitic layer. The sliders snap to the
64 configurations that have CST ground truth, so every design shows a true overlay.

In [ ]:
%matplotlib inline
# --- interactive patch designer (parameters -> geometry -> graph -> prediction vs CST)
from data.generation.create_classic_square_patch import (FEED_EDGES, make_patch_matrix,
                                                         make_reflector_matrix,
                                                         create_parameter_dict)
from src.geometry.create_pixel_antenna import create_pixel_ant, create_reflector_matrix
from src.geometry.mesh_functions import decompose_pyg_graph
from src.geometry.mesh_functions_pytorch_2 import merge_and_connect_graphs_from_dict
from src.graph.graph_functions import graph_dict_merging_duplicate_nodes

OFFSETS, PATCH_SIZE = [0, 3, 6, 9], 7
ENV = create_parameter_dict({})
_DESIGN_CACHE = {}

def build_patch_graph(row_off, col_off, feed_edge, parasitic):
    """Parameters -> model-ready graph, mirroring PixelDataset.load_pixel_data."""
    matrix = make_patch_matrix(16, PATCH_SIZE, row_off, col_off, feed_edge)
    g, _ = create_pixel_ant(matrix, ENV['threshold'], ENV['patch_x'], ENV['ground_x'],
                            ENV['ground_x'], ENV['h'], create_physical_pixel_mesh=True)
    if parasitic:
        refl = make_reflector_matrix(16, PATCH_SIZE + 2)
        gd = decompose_pyg_graph(g)
        gd['Reflector'] = create_reflector_matrix(refl, ENV, ENV['threshold'])
        g = merge_and_connect_graphs_from_dict(gd, 'sphere', k=2, add_sphere=False)
    gd = graph_dict_merging_duplicate_nodes(decompose_pyg_graph(g))
    g = merge_and_connect_graphs_from_dict(gd, 'sphere', k=2, add_sphere=False)
    g = PE_TRANSFORM(g)
    g, _ = prepare_graph(g, CFG)
    g.x = torch.cat([g.pos, g.node_normals, g.node_type], dim=-1)
    return matrix, g

def show_design(row_off=6, col_off=6, feed_edge='left', parasitic=False):
    key = (row_off, col_off, feed_edge, parasitic)
    if key not in _DESIGN_CACHE:
        matrix, g = build_patch_graph(*key)
        pred, _ = predict(MODEL, g)
        _DESIGN_CACHE[key] = (matrix, pred)
    matrix, pred = _DESIGN_CACHE[key]

    corpus = 'classic_patch_with_reflector' if parasitic else 'classic_patch_no_reflector'
    gt_id = (OFFSETS.index(row_off) * 4 + OFFSETS.index(col_off)) * 4 + FEED_EDGES.index(feed_edge)
    gt = torch.load(f'data/corpora/{corpus}/processed/processed_data_{gt_id}.pt',
                    weights_only=False)['farfeilds'][IDX_FREQ]
    m = metrics_of(pred, gt)

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
    axes[0].imshow(matrix, cmap='gray'); axes[0].set_title(
        f"patch mask{' + parasitic' if parasitic else ''}\nfeed: {feed_edge}")
    for ax, img, title, vmax in [(axes[1], pred, 'prediction', 10),
                                 (axes[2], gt, 'CST ground truth', 10),
                                 (axes[3], (pred - gt).abs(), 'absolute error', 3)]:
        im = ax.imshow(img, cmap='jet', vmin=0, vmax=vmax); ax.set_title(title)
        plt.colorbar(im, ax=ax, fraction=0.046)
    for ax in axes: ax.axis('off')
    fig.suptitle(f"MSE {m['MSE']:.3f} | MAE {m['MAE']:.3f} | MS-SSIM {m['mssim']:.3f}", y=1.04)
    plt.tight_layout(); plt.show()

try:
    import ipywidgets as widgets
    from IPython.display import display
    controls = dict(
        row_off=widgets.SelectionSlider(options=OFFSETS, value=6, description='row offset'),
        col_off=widgets.SelectionSlider(options=OFFSETS, value=6, description='col offset'),
        feed_edge=widgets.Dropdown(options=FEED_EDGES, value='left', description='feed edge'),
        parasitic=widgets.Checkbox(value=False, description='parasitic layer'))
    ui = widgets.HBox(list(controls.values()))
    display(ui, widgets.interactive_output(show_design, controls))
except Exception as err:                       # headless execution (e.g. nbconvert)
    print(f'ipywidgets unavailable ({err}); rendering the default design statically:')
show_design()   # always render one design so the saved notebook shows a result

In [ ]:
# --- zero-shot mini-benchmark: all 64 canonical square patches (~1-2 min on CPU, cached)
from pathlib import Path

def eval_corpus(corpus, model, cfg=None, cache={}):
    key = (corpus, id(model))
    if key not in cache:
        rows = []
        files = sorted(Path(f'data/corpora/{corpus}/processed').glob('processed_data_*.pt'),
                       key=lambda p: int(p.stem.split('_')[-1]))
        for f in tqdm(files, desc=corpus):
            ex = torch.load(f, weights_only=False)
            pred, _ = predict(model, graph_from_sample(ex, cfg), cfg)
            rows.append(metrics_of(pred, ex['farfeilds'][IDX_FREQ]))
        cache[key] = pd.DataFrame(rows)
    return cache[key]

zs = eval_corpus('classic_patch_no_reflector', MODEL).mean()
table = pd.DataFrame({
    'this run (n=64)': [zs['MAE'], zs['MSE'], zs['mssim'], zs['SNR']],
    'published row':   [0.1530,    0.0557,    0.9705,      24.756],
}, index=['MAE', 'MSE', 'MS-SSIM', 'PSNR (dB)'])
display(table.round(4))
print('Zero-shot square-patch metrics should reproduce the published row almost exactly '
      '(same checkpoint, same 64 examples).')

## Tier 2 — why it works

**PAIS** adds an auxiliary head that regresses the complex surface current (Re/Im of Jx, Jy,
Jz) at every mesh node, supervised by CST's currents. The current distribution is the physical
intermediate between geometry and radiation — forcing the network to get it right shapes
representations that transfer better than pattern-only training.

**The evaluation split is adversarial.** Instead of a random split, a PCA-extrapolation split
puts the far-field patterns with the most extreme first-principal-component scores into the
test set — the model is always tested on patterns *outside* the bulk of what it saw.

Below we evaluate the 20 bundled held-out PCA-test antennas against the published full-split
row, then make the PAIS ablation visible.

In [ ]:
# --- 20 held-out PCA-test antennas vs the published full-split row (~1 min on CPU, cached)
if 'TEST20' not in globals():
    TEST20 = []
    for f in tqdm(sorted(Path('data/demo/pca_test_20').glob('sample_*.pt')), desc='pca_test_20'):
        ex = torch.load(f, weights_only=False)
        TEST20.append({'name': f.stem,
                       'g': graph_from_sample(ex),
                       'label': ex['farfeilds'][IDX_FREQ],
                       'mask': ex['example_paramters']['ant_parameters']['matrix']})

def eval_models_on_test20(models, cache={}):
    """models: {column_name: (model, cfg)} -> (mean-metric table, per-model predictions)"""
    tab, preds = {}, {}
    for name, (model, cfg) in models.items():
        if name not in cache:
            rows, plist = [], []
            for s in TEST20:
                pred, _ = predict(model, s['g'], cfg)
                plist.append(pred)
                rows.append(metrics_of(pred, s['label']))
            df = pd.DataFrame(rows)
            cache[name] = ([df['MAE'].mean(), df['MSE'].mean(),
                            df['mssim'].mean(), df['SNR'].mean()], plist)
        tab[name], preds[name] = cache[name]
    return pd.DataFrame(tab, index=['MAE', 'MSE', 'MS-SSIM', 'PSNR (dB)']), preds

tab, PREDS20 = eval_models_on_test20({'GPS+PAIS (big), 20 samples': (MODEL, CFG)})
tab['published (full 3,069-sample PCA split)'] = [0.2516, 0.1831, 0.9361, 19.413]
display(tab.round(4))
print('A 20-sample subset will not match the full-split numbers exactly — expect them in range.')
assert 0.02 < tab.iloc[1, 0] < 0.8, 'MSE far outside the expected range - environment drift?'

# To evaluate the full PCA test split, download the full corpora (see data/README.md), then:
#   python -m scripts.evaluate_forward --config_file configs/forward/gps_pais_big_pca.yaml \\
#       --load_trained_model true \\
#       --trained_model_path trained_models/genial-bush-2194GNN_ff_foward_34_34_2400.pt

In [ ]:
%matplotlib inline
# --- PAIS ablation: same architecture, with vs without the surface-current head
CFG_GPS, CFG_PAIS = load_config('gps.yaml'), load_config('gps_pais.yaml')
tab, preds = eval_models_on_test20({
    'GPS (no PAIS)': (load_forward_model('trained_models/confused-smoke-2225GNN_ff_foward_34_34_2400.pt', CFG_GPS), CFG_GPS),
    'GPS + PAIS':    (load_forward_model('trained_models/lively-disco-2160GNN_ff_foward_34_34_2400.pt', CFG_PAIS), CFG_PAIS),
})
tab['published GPS'] = [0.4052, 0.4507, 0.8355, 15.186]
tab['published GPS+PAIS'] = [0.3677, 0.3875, 0.8583, 16.152]
display(tab.round(4))

fig, axes = plt.subplots(3, 4, figsize=(13, 9))
for r, s in enumerate(TEST20[:3]):
    imgs = [(s['mask'], 'antenna mask', 'gray', 1),
            (s['label'], 'CST ground truth', 'jet', 10),
            (preds['GPS (no PAIS)'][r], 'GPS (no PAIS)', 'jet', 10),
            (preds['GPS + PAIS'][r], 'GPS + PAIS', 'jet', 10)]
    for c, (img, title, cmap, vmax) in enumerate(imgs):
        axes[r, c].imshow(img, cmap=cmap, vmin=0, vmax=vmax); axes[r, c].axis('off')
        if r == 0: axes[r, c].set_title(title)
plt.tight_layout(); plt.show()
print('Note: the published GPS+PAIS row matches this specific checkpoint; the honest 3-seed '
      'mean is MAE .377 / MSE .401 / MS-SSIM .852 / PSNR 15.89 (REPRODUCIBILITY.md, caveat '
      '16). The PAIS gain holds across seeds.')

### Direction-conditioned attention: which mesh nodes radiate where?

The `+Dir+Phys` model (GPSDCC) decodes the far field with **cross-attention**: each of the
34×34 output pixels is a query built from its direction (k-vector and angles of (θ, φ)), and
it attends over all mesh-node embeddings — optionally concatenated with the PAIS-predicted
currents. That means we can ask a physically meaningful question: *for a given radiation
direction, which parts of the antenna does the model look at?* If the physics motivation is
real, the maps should resemble the phase-weighted current contributions of the radiation
integral. The repo discards attention weights (`need_weights=False`), so we capture them with
a small wrapper — no source changes needed.

In [ ]:
# --- load GPSDCC (+Dir+Phys checkpoint) and capture its direction->node attention weights
CFG_DCC = load_config('gps_pais_dir_phys.yaml')
MODEL_DCC = build_graph_model(CFG_DCC, DEVICE)
MODEL_DCC.load_state_dict(torch.load('trained_models/crisp-totem-2365_34_34_2400.pt',
                                     map_location=DEVICE))
MODEL_DCC.eval()

ATTN = {}
_orig_attn = MODEL_DCC.direction_cross_attn.forward
def _capture(query, key, value, **kw):
    kw.update(need_weights=True, average_attn_weights=False)
    out, w = _orig_attn(query, key, value, **kw)
    ATTN['w'] = w.detach()
    return out, None
MODEL_DCC.direction_cross_attn.forward = _capture

ATTN_SAMPLE = TEST20[0]
PRED_DCC, _ = predict(MODEL_DCC, ATTN_SAMPLE['g'], CFG_DCC)
print('captured attention:', tuple(ATTN['w'].shape),
      '= [batch, heads, 34*34 direction queries, mesh nodes]')

In [ ]:
%matplotlib inline
# --- attention maps: pick a direction (theta, phi), color mesh nodes by attention weight
w = ATTN['w'][0].mean(0)                                # [1156, N], averaged over 4 heads
pos = ATTN_SAMPLE['g'].pos.numpy()
theta = np.linspace(0, np.pi, 34)
phi = np.linspace(-np.pi, np.pi, 34)
picks = [(17, 17), (8, 17), (25, 8)]                    # (theta index, phi index)

fig = plt.figure(figsize=(15, 8.5))
for k, (ti, pj) in enumerate(picks):
    ax = fig.add_subplot(2, 3, k + 1)
    ax.imshow(PRED_DCC, cmap='jet', vmin=0, vmax=10)
    ax.scatter([pj], [ti], marker='x', s=120, c='white', linewidths=3)
    ax.set_title(f'direction θ={np.degrees(theta[ti]):.0f}°, φ={np.degrees(phi[pj]):.0f}°')
    ax.axis('off')

    ax3 = fig.add_subplot(2, 3, 4 + k, projection='3d')
    ax3.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=w[ti * 34 + pj].numpy(),
                cmap='viridis', s=8)
    ax3.set_title('attention over mesh nodes')
    ax3.set_box_aspect((1, 1, 0.5)); ax3.set_axis_off()
plt.suptitle('GPSDCC direction-conditioned decoder: predicted pattern (x marks the queried '
             'direction) and where that query attends', y=0.99)
plt.tight_layout(); plt.show()

### What Tiers 1–2 did and did not reproduce

Reproduced with released checkpoints: the zero-shot square row (n = 64, exactly the published
evaluation) and the PAIS-vs-no-PAIS gap. Indicative only: the 20-sample PCA-test metrics —
they use the paper's metric code and checkpoints but 20 of 3,069 test antennas. The paper's
"PSNR" column is the SNR computed by `src/losses/losses.py:compute_metrics` (see
REPRODUCIBILITY.md, caveat 12, for the two metric paths). Per-checkpoint reference numbers
ship in the released bundle's `metrics.json` files.

## Tier 3 — inverse design (needs a GPU)

Switch to a GPU runtime now: **Runtime ▸ Change runtime type ▸ T4 GPU**. The runtime restarts:
re-run the setup cells (2–5) — everything is cached, ~30 s — then continue here. Without a GPU
these cells skip themselves (CPU sampling would take 30+ min, and parts of the diffusion stack
assume CUDA).

The task: given a **target radiation pattern** (here: one of the 10 *hardest* held-out
patterns — the PCA-test patterns farthest from anything in training), sample 100 candidate
16×16 geometries (patch + parasitic channels) from a conditional diffusion U-Net with
classifier-free guidance, rebuild each as a mesh graph, and rank them with the GPS+PAIS
surrogate. **Everything here is surrogate-scored — no CST in the loop.** The paper's Table 2
re-simulates the top-5 in CST for validation; those are the numbers to trust.

In [ ]:
%matplotlib inline
# --- Tier 3 setup: diffusion U-Net + GPS+PAIS surrogate (GPU only)
import os
RUN_TIER3 = torch.cuda.is_available() or os.environ.get('DEMO_FORCE_TIER3') == '1'
if not RUN_TIER3:
    print('No GPU detected — skipping Tier 3. Switch to a T4 runtime and re-run from cell 2.')
else:
    inv_cfg = _load_yaml('configs/base_inverse.yaml')
    _inv_defaults = {   # the paper's run defaults (src/diffusion/diffusion_trainer.py), demo-sized where noted
        'guidance_scale': 2.0,                  # manual-target CFG scale (paper Fig. 3)
        'num_hidden': 128, 'batch_size': 128, 'num_epochs': 1000, 'lr': 1e-4,
        'img_wh': (16, 16), 'eval_batch_size': 20, 'number_of_best_samples': 5,
        'number_of_samples_to_generate': 100,   # demo-sized (paper: 500)
        'T': 700, 'num_classes': 4096, 'idx_freq': 3,
        'raw_node_feture_size': 17, 'radiation_image_shape': [34, 34, 1],
        'threshold': 0.5, 'physical_antenna': True, 'use_node_probs': False,
        'add_sphere': False, 'merge_duplicate_nodes': True,
        'frequencies': [2400, 2800, 5200, 5600, 6000],
        'frequencies_in_scale': [2.4e9, 2.8e9, 5.2e9, 5.6e9, 6.0e9],
        'train_freq_idxs': [0, 1, 2, 3, 4],
        'data_set_type': 'pixel_data_with_reflectors',
        'add_radius_graph_edges': True, 'predict_surface_current': True,
        'plot_epochs': [],
    }
    for k, v in _inv_defaults.items():
        inv_cfg.setdefault(k, v)
    inv_cfg.update(number_of_samples_to_generate=100, guidance_scale=2.0,
                   output_dir='outputs/demo_inverse/')

    from src.models.gps import GPS
    from src.diffusion.unet_attention import ConditionalDenoisingUNetSmallAttentionWithConcat
    from src.diffusion.diffusion_trainer import DiffusionTrainer   # forces the Agg backend on import
    import matplotlib
    matplotlib.use('module://matplotlib_inline.backend_inline', force=True)

    GPU = torch.device('cuda')
    surrogate = GPS(config=inv_cfg, in_channels=16, channels=128, pe_dim=10, num_layers=10,
                    attn_type='multihead', attn_kwargs={'dropout': 0.3},
                    out_channels=34 * 34).to(GPU)
    unet = ConditionalDenoisingUNetSmallAttentionWithConcat(
        config=inv_cfg, in_channels=2, num_hiddens=inv_cfg['num_hidden'], ff_in_ch=1)
    unet.load_state_dict(torch.load('checkpoints/diffusion/diffusion_model.pt',
                                    map_location=GPU))
    trainer = DiffusionTrainer(inv_cfg, unet, surrogate)  # also loads the surrogate checkpoint
    print('diffusion U-Net + surrogate ready on', torch.cuda.get_device_name(0))

In [ ]:
# --- generate 100 candidates for one hard target and rank them (~2-4 min on T4, cached)
TARGET_RANK = 0   # <-- pick 0-9: which of the 10 hardest held-out targets (0 = hardest)

if RUN_TIER3:
    import glob
    target_path = sorted(glob.glob(f'data/demo/manual_targets_hard10/hard_{TARGET_RANK:02d}_*.pt'))[0]
    if 'INV' in globals() and INV['target_name'] == Path(target_path).name:
        print(f"using cached candidates for {INV['target_name']} — change TARGET_RANK to regenerate")
    else:
        target = torch.load(target_path, weights_only=False).float()
        t0 = time.time()
        # the U-Net expects the condition as (N, 1, 34, 34)
        labels = target[None, None].expand(inv_cfg['number_of_samples_to_generate'], 1, 34, 34).to(GPU)
        samples = trainer.ddpm.sample(c=labels, img_wh=inv_cfg['img_wh'],
                                      guidance_scale=inv_cfg['guidance_scale'], in_channels=2)
        t_sample = time.time() - t0
        preds, gts, losses, antennas, valid = trainer.pred_batch_ff(samples, labels)
        INV = dict(target=target, target_name=Path(target_path).name, samples=samples.cpu(),
                   preds=preds, losses=losses, valid=valid)
        print(f'{Path(target_path).name}: sampled {len(labels)} candidates in {t_sample:.0f} s '
              f'({inv_cfg["T"]} DDPM steps), surrogate-ranked in {time.time()-t0-t_sample:.0f} s')

In [ ]:
%matplotlib inline
# --- top-5 candidates vs the target, next to the nearest-neighbor retrieval baseline
if RUN_TIER3:
    from src.diffusion.smooth_binarize import smooth_binarize
    order = np.argsort(INV['losses'])[:5]
    nn = torch.load('data/demo/nn_precomputed/nn_results.pt',
                    weights_only=False)[INV['target_name']]

    fig, axes = plt.subplots(2, 7, figsize=(21, 6))
    axes[0, 0].imshow(INV['target'], cmap='jet', vmin=0, vmax=10)
    axes[0, 0].set_title('TARGET pattern\n(hardest held-out set)')
    axes[1, 0].set_visible(False)
    for col, k in enumerate(order, start=1):
        oi = INV['valid'][k]
        mask = torch.cat([smooth_binarize(INV['samples'][oi, 0], 300, 300),
                          smooth_binarize(INV['samples'][oi, 1], 300, 300)], dim=1)
        axes[0, col].imshow(INV['preds'][k].reshape(34, 34), cmap='jet', vmin=0, vmax=10)
        axes[0, col].set_title(f"rank {col}\nsurrogate MSE {INV['losses'][k]:.3f}")
        axes[1, col].imshow(mask, cmap='gray')
        axes[1, col].set_title('mask (patch | parasitic)', fontsize=9)
    nn_mask = (torch.cat([nn['nn_matrix'], nn['nn_reflector']], dim=1)
               if nn['nn_reflector'] is not None else nn['nn_matrix'])
    axes[0, 6].imshow(nn['nn_farfield'], cmap='jet', vmin=0, vmax=10)
    axes[0, 6].set_title(f"NN retrieval baseline\nMSE {nn['nn_mse_clamped']:.3f}")
    axes[1, 6].imshow(nn_mask, cmap='gray')
    axes[1, 6].set_title('retrieved training antenna', fontsize=9)
    for ax in axes.flat:
        ax.axis('off')
    plt.tight_layout(); plt.show()
    print('CAVEAT: candidate patterns and rankings above come from the GPS+PAIS surrogate, '
          'not CST. The paper validates the top-5 in CST (Table 2); the target-column NN '
          'baseline here retrieves the closest *training* pattern for reference.')

In [ ]:
%matplotlib inline
# --- watch the diffusion: denoising trajectory of 8 candidates as a GIF (~40 s on T4)
if RUN_TIER3:
    import imageio.v2 as imageio
    from IPython.display import Image as IPImage, display

    sched, unet_, num_ts = trainer.ddpm.ddpm_schedule, trainer.ddpm.unet, trainer.ddpm.num_ts
    n = 8
    c = INV['target'][None, None].expand(n, 1, 34, 34).to(GPU)
    frames = []
    with torch.inference_mode():                    # mirrors src/diffusion/ddpm.py:ddpm_cfg_sample
        x_t = torch.randn(n, 2, 16, 16, device=GPU)
        for t in range(num_ts - 1, 0, -1):
            z = torch.randn_like(x_t) if t > 0 else torch.zeros_like(x_t)
            t_norm = torch.ones(n, 1, device=GPU) * (t / num_ts)
            pred_c, pred_u = unet_(x_t, t=t_norm, c=c), unet_(x_t, t=t_norm, c=torch.zeros_like(c))
            pred = pred_u + inv_cfg['guidance_scale'] * (pred_c - pred_u)
            a_t, ab_t, b_t = sched['alphas'][t], sched['alpha_bars'][t], sched['beta_array'][t]
            ab_prev = sched['alpha_bars'][t - 1]
            x0_hat = (x_t - (1 - ab_t).sqrt() * pred) / ab_t.sqrt()
            x_t = ((ab_prev.sqrt() * b_t / (1 - ab_t)) * x0_hat
                   + (a_t.sqrt() * (1 - ab_prev) / (1 - ab_t)) * x_t + b_t.sqrt() * z)
            if t % 25 == 1 or t == 1:
                grid = (x_t[:, 0].reshape(2, 4, 16, 16).permute(0, 2, 1, 3)
                        .reshape(32, 64).cpu().numpy())
                frames.append(grid)

    imgs = []
    for f in frames:
        f01 = (f - f.min()) / (f.max() - f.min() + 1e-9)
        imgs.append((plt.cm.gray(np.kron(f01, np.ones((6, 6))))[..., :3] * 255).astype(np.uint8))
    imgs += [imgs[-1]] * 6                          # hold the final frame
    imageio.mimsave('denoising.gif', imgs, fps=8)
    print('patch channel of 8 candidates, from pure noise (t=699) to antennas (t=1):')
    display(IPImage('denoising.gif'))

## Going further

Everything here used released checkpoints; the training code lives in the repo:

- **Forward training / evaluation** — `scripts/train_forward.py` + `configs/forward/*.yaml`;
  each Table-1 row: `bash experiments/table1.sh <row>`. Graph baselines (GCN, GAT,
  Graph-U-Net, DGCNN, MeshGraphNets) each have a paired `*_pais` config.
- **Inverse design** — `scripts/run_inverse.py` + `configs/inverse/*.yaml`;
  `experiments/table2.sh` / `table3.sh` (export candidates → manual CST validation →
  `scripts/evaluate_cst.py`).
- **Data generation** — `data/generation/` builds every corpus geometry;
  [data/README.md](https://github.com/AviEpstein/GNN-for-Antenna-design/blob/public/data/README.md)
  documents the CST simulation setup for creating ground truth (~530 GB for the full 80k-sample
  dataset; hosting in preparation).
- **Reproducibility** —
  [REPRODUCIBILITY.md](https://github.com/AviEpstein/GNN-for-Antenna-design/blob/public/REPRODUCIBILITY.md)
  maps every table and figure to exact commands and documents 19 known caveats.

## Citation

```bibtex
@inproceedings{epstein2026physics,
  title     = {Physics-Augmented Graph Transformers for Patch-Antenna Forward and Inverse Design},
  author    = {Epstein, Avi and Nehemia, Snir and Suchowski, Haim and Wolf, Lior},
  booktitle = {2026 IEEE International Workshop on Machine Learning for Signal Processing (MLSP)},
  year      = {2026},
  publisher = {IEEE}
}
```

Licensed under MIT. Full dataset and checkpoint hosting: links will appear in the repo README
when ready. Questions and issues: please use the
[issue tracker](https://github.com/AviEpstein/GNN-for-Antenna-design/issues).